# Kubernetes Pod Eviction Risk Prediction

**Assignment**: Multi-class classification to predict pod eviction risk (low/medium/high) using synthetic Kubernetes metrics.

**Dataset**: Pre-generated from a Minikube cluster under controlled resource pressure. Features include CPU/memory requests, limits, priority, node pressure metrics, and pod usage.

**Goal**: Train a RandomForest classifier, evaluate with standard metrics, and demonstrate predictions on three example inputs.

---

This notebook is designed to run on mybinder.org without requiring Kubernetes or data generation steps.

## 1. Environment Setup and Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display versions
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")

## 2. Load Pre-Generated Dataset

The dataset was generated from a Minikube cluster under controlled memory pressure. Each row represents a pod snapshot after observation.

In [ ]:
# Load dataset
DATA_PATH = Path("data/pod_risk_data_fast_combined.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Please ensure the CSV is in the repo.")

df = pd.read_csv(DATA_PATH)

print(f"✅ Loaded {len(df)} samples from {DATA_PATH}")
print(f"\n📊 Class distribution:\n{df['risk'].value_counts()}")
print(f"\n🔍 Dataset shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")

## 3. Prepare Features and Target

We'll use features that don't directly encode the heuristic labeling rule (dropping `pod_mem_usage_mi` and `mem_limit_mi` to avoid leakage).

In [ ]:
# Define features (excluding leakage columns for heuristic labels)
LEAKY_FEATURES = ['pod_mem_usage_mi', 'mem_limit_mi']
BASE_FEATURES = [
    'cpu_request_m', 'cpu_limit_m', 'mem_request_mi',
    'priority', 'node_cpu_pressure_pct', 'node_mem_pressure_pct',
    'pod_cpu_usage_pct'
]

# Filter to available columns
feature_cols = [c for c in BASE_FEATURES if c in df.columns]
print(f"ℹ️  Using features: {feature_cols}")
print(f"ℹ️  Dropped leakage features: {LEAKY_FEATURES}")

X = df[feature_cols]
y = df['risk']

print(f"\n📐 Feature matrix shape: {X.shape}")
print(f"🎯 Target distribution:\n{y.value_counts(normalize=True).round(3)}")

## 4. Train/Test Split

Split data into 80% training and 20% test sets with stratification to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y
)

print(f"✅ Train set: {X_train.shape[0]} samples")
print(f"✅ Test set: {X_test.shape[0]} samples")
print(f"\n📊 Train set class distribution:\n{y_train.value_counts()}")
print(f"\n📊 Test set class distribution:\n{y_test.value_counts()}")

## 5. Define and Train Model

Using RandomForestClassifier with balanced class weights to handle class imbalance.

In [ ]:
# Create pipeline with scaling and Random Forest
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("🌲 Training Random Forest model...")
pipeline.fit(X_train, y_train)
print("✅ Training complete!")

## 6. Evaluation Metrics

Evaluate model performance on the test set with classification metrics.

In [ ]:
# 6. Evaluation Metrics
from sklearn.metrics import classification_report, accuracy_score

# Generate predictions
y_pred = pipeline.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Test Accuracy: {accuracy:.2%}\n")

# Detailed classification report
print("📋 Classification Report:")
print(classification_report(y_test, y_pred))

## 6a. Extended Metrics (Balanced Accuracy, Macro/Weighted Aggregates, ROC-AUC, Binary Breakdown)
We compute additional evaluation metrics required by the assignment:
- Balanced accuracy (accounts for class imbalance)
- Macro & weighted precision/recall/F1 (already in classification_report, but we summarize)
- Multiclass ROC-AUC using One-vs-Rest (OVR) and One-vs-One (OVO) strategies
- Binary confusion matrix treating 'high' as positive and (low|medium) as negative for TP/TN/FP/FN counts.

In [ ]:
# Extended metrics computation
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, precision_recall_fscore_support
import numpy as np

# Balanced accuracy
balanced_acc = balanced_accuracy_score(y_test, y_pred)

# Precision/Recall/F1 macro & weighted (redundant with classification_report but summarized)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

# Multiclass ROC-AUC (requires probability estimates)
y_proba = pipeline.predict_proba(X_test)
auc_ovr_macro = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
auc_ovo_macro = roc_auc_score(y_test, y_proba, multi_class='ovo', average='macro')
auc_ovr_weighted = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
auc_ovo_weighted = roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted')

# Binary breakdown for 'high' vs 'non-high'
y_test_high = (y_test == 'high')
y_pred_high = (y_pred == 'high')
TP = np.sum((y_test_high) & (y_pred_high))
TN = np.sum((~y_test_high) & (~y_pred_high))
FP = np.sum((~y_test_high) & (y_pred_high))
FN = np.sum((y_test_high) & (~y_pred_high))

print("🔧 Extended Metrics:")
print(f"Balanced Accuracy: {balanced_acc:.3f}")
print(f"Macro Precision: {prec_macro:.3f} | Macro Recall: {rec_macro:.3f} | Macro F1: {f1_macro:.3f}")
print(f"Weighted Precision: {prec_weighted:.3f} | Weighted Recall: {rec_weighted:.3f} | Weighted F1: {f1_weighted:.3f}")
print(f"ROC-AUC OVR Macro: {auc_ovr_macro:.3f} | ROC-AUC OVO Macro: {auc_ovo_macro:.3f}")
print(f"ROC-AUC OVR Weighted: {auc_ovr_weighted:.3f} | ROC-AUC OVO Weighted: {auc_ovo_weighted:.3f}")
print("\nBinary (high vs non-high) confusion components:")
print(f"TP: {TP} | TN: {TN} | FP: {FP} | FN: {FN}")

## 7. Confusion Matrix

Visualize classification performance across all classes.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['high', 'medium', 'low'])

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['high', 'medium', 'low'])
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix - Pod Eviction Risk Prediction')
plt.tight_layout()
plt.show()

print("\n📊 Confusion Matrix (rows=actual, cols=predicted):")
print(cm)

## 8. Feature Importance

Analyze which features contribute most to the predictions.

In [ ]:
# Extract feature importances from the Random Forest
rf_model = pipeline.named_steps['classifier']
importances = rf_model.feature_importances_

# Create DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print("🔍 Feature Importances:")
print(feature_importance_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance_df['feature'], feature_importance_df['importance'])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - Random Forest Model')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Example Predictions

Demonstrate the model with three sample inputs representing low, medium, and high eviction risk scenarios.

In [ ]:
# Simplified example predictions: detailed lines + summary (no function/flags)
if 'pipeline' not in globals() or 'feature_cols' not in globals():
    raise RuntimeError('Required variables missing: run training & feature prep cells first.')

examples = [
    {
        'name': 'Low Risk Pod',
        'description': 'Low resource usage, low node pressure',
        'features': {
            'cpu_request_m': 100,
            'cpu_limit_m': 200,
            'mem_request_mi': 128,
            'priority': 0,
            'node_cpu_pressure_pct': 10.0,
            'node_mem_pressure_pct': 15.0,
            'pod_cpu_usage_pct': 20.0
        }
    },
    {
        'name': 'Medium Risk Pod',
        'description': 'Moderate resource usage, moderate node pressure',
        'features': {
            'cpu_request_m': 500,
            'cpu_limit_m': 1000,
            'mem_request_mi': 512,
            'priority': 0,
            'node_cpu_pressure_pct': 60.0,
            'node_mem_pressure_pct': 70.0,
            'pod_cpu_usage_pct': 75.0
        }
    },
    {
        'name': 'High Risk Pod',
        'description': 'High resource usage, high node pressure',
        'features': {
            'cpu_request_m': 1000,
            'cpu_limit_m': 2000,
            'mem_request_mi': 1024,
            'priority': 0,
            'node_cpu_pressure_pct': 85.0,
            'node_mem_pressure_pct': 90.0,
            'pod_cpu_usage_pct': 95.0
        }
    }
]
assert len(examples) == 3, f"Unexpected number of examples: {len(examples)}"

rows = []
print("🔮 Example Predictions\n")
for i, ex in enumerate(examples, 1):
    input_df = pd.DataFrame([ex['features']])[feature_cols]
    pred = pipeline.predict(input_df)[0]
    probs = pipeline.predict_proba(input_df)[0]
    class_probs = dict(zip(pipeline.classes_, probs))
    prob_sum = float(probs.sum())
    prob_str = (
        f"high={class_probs.get('high', 0):.2%}, "
        f"medium={class_probs.get('medium', 0):.2%}, "
        f"low={class_probs.get('low', 0):.2%}"
    )
    print(f"{i}. {ex['name']}")
    print(f"   Description: {ex['description']}")
    print(f"   Input: {ex['features']}")
    print(f"   ⚠️  Predicted Risk: {pred.upper()}")
    print(f"   Probabilities: {prob_str} (sum={prob_sum:.3f})")
    print()
    rows.append({
        'example': i,
        'name': ex['name'],
        'predicted': pred,
        'high_prob': class_probs.get('high', 0.0),
        'medium_prob': class_probs.get('medium', 0.0),
        'low_prob': class_probs.get('low', 0.0),
        'prob_sum': prob_sum
    })

summary_df = pd.DataFrame(rows)
summary_df['prob_sum_ok'] = (summary_df['prob_sum'].round(3) - 1.0).abs() < 1e-3
print("📊 Example Predictions Summary:")
print(summary_df.to_string(index=False, formatters={
    'high_prob': '{:.3f}'.format,
    'medium_prob': '{:.3f}'.format,
    'low_prob': '{:.3f}'.format,
    'prob_sum': '{:.3f}'.format
}))
print("\n✅ Finished generating predictions for 3 scenarios.")

## 10. Model Persistence

Save the trained model for future use.

In [ ]:
# Save model
model_path = 'pod_risk_model.pkl'
joblib.dump(pipeline, model_path)
print(f"💾 Model saved to: {model_path}")

# Verify by loading
loaded_model = joblib.load(model_path)
test_prediction = loaded_model.predict(X_test[:1])
print(f"✅ Model loaded successfully, test prediction: {test_prediction[0]}")

## 11. Summary

This notebook demonstrated a complete ML pipeline for predicting Kubernetes pod eviction risk.

**Key Pipeline Stages:**
1. Data Loading (320 samples; classes: high=143, low=141, medium=36)
2. Feature Preparation (7 non-leaky features retained; dropped `pod_mem_usage_mi`, `mem_limit_mi`)
3. Train/Test Split (256 train / 64 test, stratified)
4. Model (RandomForest, 200 estimators, `class_weight='balanced'`)
5. Core Metrics (Accuracy: 53.12%)
6. Extended Metrics: Balanced Accuracy 0.433; Macro P/R/F1 ≈ 0.44 / 0.43 / 0.43; Weighted F1 ≈ 0.52; Macro ROC-AUC (OVR/OVO): 0.667 / 0.657; Weighted ROC-AUC (OVR/OVO): 0.665 / 0.661
7. Confusion Matrix (3-class) plus binary breakdown (high vs non-high: TP=19, TN=20, FP=15, FN=10)
8. Feature Importance (Top: `pod_cpu_usage_pct`, node pressure metrics)
9. Scenario Predictions (Low/Medium/High examples)
10. Model Persistence (saved to `pod_risk_model.pkl`)

**Observations:**
- Strong class imbalance (medium only 11%) depresses macro metrics.
- High false positives for 'high' risk (FP=15) indicate need for threshold tuning or better features.
- CPU usage and node pressure dominate importance—consistent with synthetic generation logic.
- ROC-AUC scores (>0.65 macro) show moderate separability despite imbalance.

**Improvement Ideas:**
- Collect real eviction events for ground-truth labels.
- Add temporal / lifecycle features (age, restarts), memory working-set ratios, network I/O.
- Try class imbalance strategies (SMOTE, focal loss, or adjusting class weights further).
- Evaluate gradient boosting (XGBoost/LightGBM) and probability calibration.
- Consider cost-sensitive evaluation: penalize high-risk false negatives more heavily.